# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [4]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [5]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [11]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [12]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [13]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [14]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 2 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [15]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://huggingface.co/'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Zhihu', 'url': 'https://www.zhihu.com/org/huggingface'},
  {'type': 'Discord / Community',
   'url': 'https://huggingface.co/join/discord'},
  {'type': 'Community forum', 'url': 'https://discuss.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [16]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [17]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
google/gemma-4-31B-it
Updated
7 days ago
•
1.11M
•
1.47k
dealignai/Gemma-4-31B-JANG_4M-CRACK
Updated
5 days ago
•
44.2k
•
788
zai-org/GLM-5.1
Updated
about 13 hours ago
•
1.3k
•
742
netflix/void-model
Updated
2 days ago
•
646
google/gemma-4-26B-A4B-it
Updated
7 days ago
•
836k
•
541
Browse 2M+ models
Spaces
Running
on
Zero
Featured
291
OmniVoice
🌍
291
High-quality voice cloning TTS for 600+ languages
Running
on
Zero
MCP
1.93k
Wan2.2 14B Preview
🐌
1.93k
generate a video from an image with a text prompt
Running
on
Zero
MCP
Featured
7

In [26]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [19]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [20]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ngoogle/gemma-4-31B-it\nUpdated\n7 days ago\n•\n1.11M\n•\n1.47k\ndealignai/Gemma-4-31B-JANG_4M-CRACK\nUpdated\n5 days ago\n•\n44.2k\n•\n788\nzai-org/GLM-5.1\nUpdated\nabout 13 hours ago\n•\n1.3k\n•\n742\nnetflix/void-model\nUpdated\n2 days ago\n•\n646\ngoogle/gemma-4-26B-A4B-it\nUpdated\n7 days ago\n•\n836k\n•\n541\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n291\nOmniVoice\n🌍

In [21]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [22]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the leading collaboration platform for the machine learning (ML) community. It serves as a central hub where ML engineers, scientists, and enthusiasts from around the world share, explore, and experiment with open-source machine learning models, datasets, and applications. Hugging Face empowers the next generation of AI innovators to build an open, ethical, and collaborative AI future.

---

## What We Offer

- **Models:** Access and share over 2 million machine learning models covering diverse modalities such as text, image, video, audio, and 3D.
- **Datasets:** Browse from more than 500,000 datasets contributed and curated by the community.
- **Spaces:** Build and explore interactive AI applications hosted on the platform.
- **Enterprise Plans:** Scalable solutions with enterprise-grade security, advanced compute options, single sign-on (SSO), access control, private datasets management, and analytics for teams and organizations.
- **Open Source Tools:** Leverage one of the most popular and fastest-growing ML open-source stacks to build and deploy AI applications faster.
- **Community:** Join a vibrant and fast-growing AI community collaborating on cutting-edge research and development.

---

## Company Culture

At Hugging Face, collaboration and openness are core values. The company fosters a culture where learning, sharing, and ethical AI development are paramount. The team is passionate about democratizing AI technology, encouraging innovation, and creating accessible tools that enable anyone to build and share machine learning projects in a global community.

Hugging Face believes in:
- Transparent and ethical AI advancement
- Empowering developers through open source
- Supporting a diverse and inclusive machine learning community
- Continuous learning and cutting-edge scientific research

---

## For Customers and Enterprise Users

Hugging Face offers flexible plans tailored for individuals, teams, and large enterprises, providing:

- **Team & Enterprise Plans:** Starting at $20 per user per month, teams can collaborate securely with Single Sign-On (SSO), manage data location and audit logs, granular access control, token management, and usage analytics.
- **Advanced Compute Options:** Including GPU resources with zero upfront cost quotas scaled for organizational needs.
- **Private Storage & Dataset Viewer:** Additional private storage space and tools to facilitate secure collaboration and data management.
- **Dedicated Support:** Assistance and guidance for enterprise-grade deployment.

These features ensure that organizations can scale their AI projects securely and efficiently with full control and governance.

---

## Careers

Hugging Face is looking for passionate individuals to join their talented team of scientists, engineers, and community builders who are pioneering AI innovation. The company values creativity, collaboration, and a drive to push the boundaries of technology. Working here means contributing to open-source ML tools used by millions worldwide and helping shape the future of AI.

Interested candidates can explore career opportunities directly on the Hugging Face careers page.

---

## Join the AI Revolution

Become part of a dynamic and supportive community shaping the future of artificial intelligence. Whether you are a beginner eager to learn, an expert looking to contribute, or an enterprise aiming to power your AI applications with cutting-edge tools, Hugging Face provides the platform and ecosystem to accelerate your machine learning journey.

**Visit:** [huggingface.co](https://huggingface.co)  
**Connect:** GitHub, Twitter, LinkedIn, Discord

---

## Brand Identity

- **Logo Colors:** Vibrant Yellow (#FFD21E), Orange (#FF9D00), and Gray (#6B7280)
- **Mission:** Build an open, collaborative, and ethical AI community for everyone.

---

Build, share, and accelerate your machine learning projects with Hugging Face — the AI community building the future.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [23]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [27]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Welcome to Hugging Face: Your Friendly Neighborhood AI Superheroes 🤗

---

## Who Are We?

Imagine a bustling, vibrant village where the brightest minds in machine learning come together, armed not with swords, but with models, datasets, and boundless creativity. Welcome to **Hugging Face**, the AI community *actually* building the future! We're the global platform where engineers, scientists, and AI enthusiasts unite to create, share, and collaborate on cutting-edge Machine Learning projects.

### Our Mission:  
**Democratize good machine learning, one commit at a time.**  
Because the future shouldn’t be locked behind corporate doors — it belongs to *everybody*. And yes, that means YOU can join this AI party too!

---

## What Do We Offer?

- **2 Million+ Models**: From the smallest chatbot to huge language beasts like Google’s Gemma-4, there’s a model for every AI ambition.
- **500k+ Datasets**: Need data? Oh, we have that. Ready to train your own masterpiece or test your latest algorithm? Dive right in.
- **1 Million+ Applications & Spaces**: Want to experiment with voice cloning in 600+ languages? Or create videos from images with a simple text prompt? Spaces have got you covered.
- **Buckets**: For hosting and collaborating on your personal ML projects.
- **Open Source Tech Stack**: The gift that keeps on giving—tools and libraries to get your AI workflows humming faster and smoother.

---

## Why Hugging Face?

- **For the People, By the People** — We are the *collaboration platform* for the AI community. Here, you don’t just use models; you share, discover, build, and grow alongside tens of thousands of peers worldwide.
- **Open & Ethical AI** — We geek out over transparency and inclusion, supporting an AI future that’s ethical, open, and accessible to all.
- **Multimodal Magic** — Text, image, audio, video, even 3D! Whatever your flavor of AI, Hugging Face supports it.
- **Build Your Portfolio** — Showcase and share your AI wizardry; build a profile that recruiters drool over.
- **Cutting-Edge Research + Practical Tools** — From publishing new papers to launching learning tracks (hello DataCamp!), we’re at the bleeding edge of AI innovation.

---

## Meet The Team

We’re a diverse troop of 180+ passionate coders, scientists, and dreamers (and growing fast!). If you’re excited by working in a supportive, energetic environment where one commit can change the world, it might be time to hop on our bandwagon.  

Whether you code in Python pajamas or caffeinate at a desk in Paris, Berlin, or Bangalore, Hugging Face values contributions big and small. Plus, our community spirit is as contagious as our emoji game—because AI is serious, but we like to smile 😄.

---

## Join Us

- **Looking to Learn?** Explore our documentation and ever-growing catalog of models.
- **Want to Contribute?** Fork, commit, push, and share what you’ve built.
- **Hungry For a Career?** We’re hiring! Join us to democratize AI, build open ecosystems, and have fun doing it.

---

## Customers & Collaborators

From industry leaders like Google and Netflix to startups and solo researchers — they all build on Hugging Face's open platform to innovate faster. You could say: If the AI revolution had a city, we’re the town square.

---

## Fun Facts

- We’re proudly powered by yellow (#FFD21E) and orange (#FF9D00), because AI needs a little sunshine.
- Our logo? A smiling hugging face — because why make AI cold and scary?
- We love hosting AI conversations on GitHub, Twitter, LinkedIn, and Discord — we’re always just a ping away.

---

### Ready to jump into the future?

**Hugging Face** – Where AI is a community hug, not a lone wolf.

Visit us 👉 [huggingface.co](https://huggingface.co)  
Join the conversation on Discord, GitHub and beyond!

---

*Claim your spot at the AI table—fun, fierce, and ridiculously friendly.*  
See you on the Hub! 🤗

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>